<a href="https://colab.research.google.com/github/AdityaTheEmpire/Founderstuff/blob/main/Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests beautifulsoup4 pandas sentence-transformers scikit-learn hdbscan feedparser

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.2 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=6bb9414d1d603228c91dfe56d0e4c1b9e2fa2fe7b22cbcadf38cee0cc074004e
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


Finding post from reddit

In [ ]:
import requests
import csv
import time
from datetime import datetime, timezone

BASE = "https://arctic-shift.photon-reddit.com/api"

HEADERS = {
    "User-Agent": "saas-founder-pain-research/1.0",
    "Accept":     "application/json",
}

# ── ALL QUERIES (deduplicated from both query sets) ───────────
SEARCH_QUERIES = list(set([

    # MANUAL CRM DATA ENTRY COMPLAINTS
    "manual crm data entry", "manually update crm", "crm data entry takes time",
    "crm data entry takes forever", "spend time updating crm", "spend hours updating crm",
    "updating crm manually", "hate updating crm", "tired of updating crm",
    "crm data entry annoying", "crm data entry frustrating", "crm admin work",
    "crm admin work taking time", "crm busy work", "crm paperwork",
    "crm data entry every day", "crm updates every day", "crm updates after every call",
    "crm updates after meetings", "crm updates take too long", "entering data into crm",
    "entering notes into crm", "entering sales notes manually", "updating deals manually",
    "manual deal updates crm", "manual sales updates crm", "manual pipeline updates",
    "manual crm maintenance", "crm data entry sucks", "crm admin tasks annoying",
    "crm admin takes time", "crm admin instead of selling", "doing crm admin all day",
    "crm data entry workload", "crm updates slow down sales", "crm admin slowing sales",
    "manual crm entry after calls", "manual crm updates every day",
    "crm data entry repetitive", "crm updates repetitive", "constant crm updates",
    "crm data entry backlog", "crm updates backlog", "crm updates pile up",
    "crm tasks pile up", "crm updates end of day", "crm updates before leaving",
    "crm updates after work", "manual crm tracking", "crm entry after every call",

    # CALL / EMAIL / MEETING LOGGING PAIN
    "logging calls manually", "logging calls manually crm", "log calls into crm",
    "log calls after call", "log calls after meetings", "log emails into crm",
    "logging emails manually", "logging meetings manually", "logging activities manually",
    "manual activity logging", "manual sales activity logging",
    "crm activity logging takes time", "crm call logging takes time",
    "crm call logging annoying", "crm call logging tedious", "crm meeting notes entry",
    "enter meeting notes crm", "record meeting notes crm", "record call notes crm",
    "write call notes crm", "write sales notes crm", "enter call notes manually",
    "log every call crm", "log every email crm", "log every meeting crm",
    "crm activity tracking manual", "crm call notes manual", "crm email logging manual",
    "crm meeting logging manual", "crm activity entry repetitive", "log outreach manually crm",
    "log prospect calls crm", "log discovery calls crm", "log sales calls crm",
    "log follow up calls crm", "log meetings after meeting", "log calls end of day",
    "log emails end of day", "log meetings end of day", "log sales activity end of day",
    "log activities after calls", "log activity backlog crm", "log calls backlog crm",
    "crm notes entry takes time", "crm call note entry slow", "crm meeting note entry slow",
    "crm activity entry slow", "crm call logging slow",

    # PIPELINE UPDATE PAIN
    "pipeline not updated", "pipeline always outdated", "pipeline inaccurate",
    "sales pipeline inaccurate", "sales pipeline outdated", "pipeline data unreliable",
    "pipeline numbers wrong", "pipeline forecast wrong", "pipeline messy",
    "pipeline messy crm", "crm pipeline messy", "crm pipeline outdated",
    "crm pipeline inaccurate", "crm pipeline unreliable", "pipeline never updated",
    "pipeline not maintained", "pipeline maintenance crm", "pipeline updates annoying",
    "pipeline update takes time", "update deal stages manually", "update deals manually",
    "deal stages outdated", "deals not updated crm", "deals outdated crm",
    "deal tracking messy", "deal tracking difficult", "deal stages wrong",
    "pipeline status outdated", "pipeline stage wrong", "pipeline always wrong",
    "pipeline numbers inaccurate", "pipeline management manual", "pipeline tracking manual",
    "pipeline update backlog", "pipeline cleanup crm", "pipeline cleanup takes time",
    "pipeline cleanup manual", "pipeline always behind", "pipeline data behind",
    "pipeline entries outdated", "pipeline maintenance painful", "pipeline data messy",
    "pipeline updates forgotten", "pipeline updates delayed", "pipeline updates end of day",
    "pipeline updates once a week", "pipeline unreliable crm",

    # FOLLOW UP TRACKING PAIN
    "forget to follow up leads", "forgot to follow up lead", "missing follow ups",
    "follow ups slipping", "follow ups slipping through cracks",
    "lost lead because follow up", "lost deal because follow up",
    "missed follow up", "missed follow up lead", "missed follow up opportunity",
    "follow up tracking messy", "follow up tracking manual",
    "follow up reminders missing", "follow up reminders unreliable",
    "follow ups hard to track", "follow ups hard to manage",
    "too many follow ups", "follow ups piling up", "follow ups backlog",
    "follow ups forgotten", "crm follow up tracking messy", "crm follow up tracking manual",
    "crm follow up tracking difficult", "crm follow up reminders missed",
    "crm follow ups unreliable", "crm follow ups slipping", "crm follow ups forgotten",
    "crm follow up workload", "crm follow up management difficult",
    "crm follow up management messy",

    # MULTI LEAD WORKLOAD SIGNALS
    "too many leads to track", "too many leads to manage", "managing too many leads",
    "managing too many prospects", "tracking too many prospects",
    "too many deals in pipeline", "too many deals to track", "too many deals to manage",
    "pipeline too big to manage", "crm overloaded with leads", "crm overloaded with deals",
    "hard to track many leads", "hard to manage many leads",
    "keeping track of leads difficult", "keeping track of deals difficult",
    "lost track of leads", "lost track of deals", "deal tracking difficult",
    "deal tracking messy", "deal tracking confusing", "pipeline tracking difficult",
    "pipeline tracking manual", "pipeline tracking messy", "lead tracking messy",
    "lead tracking difficult", "lead tracking manual", "lead management messy",
    "lead management difficult", "lead management manual", "prospect tracking messy",
    "prospect tracking difficult", "prospect tracking manual",
    "prospect tracking crm messy", "prospect tracking crm difficult",
    "prospect tracking crm manual",

    # HUMAN BEHAVIOR SIGNALS
    "forget to update crm", "forgot to update crm", "forget to log calls crm",
    "forget to log meetings crm", "forget to log emails crm", "forget to update deals",
    "forget to update pipeline", "forget to log activity", "update crm end of day",
    "update crm at night", "update crm once a week", "update crm before meetings",
    "update crm after meetings", "update crm after calls", "crm updates delayed",
    "crm updates behind", "crm updates backlog", "crm updates pile up",
    "crm updates ignored", "crm updates forgotten", "sales reps ignore crm",
    "sales team ignores crm updates", "team forgets crm updates",
    "crm entries forgotten", "crm entries delayed", "crm entries backlog",
    "crm entries end of day", "crm entries before leaving",
    "crm entries once a week", "crm entries rarely done",

    # CONSEQUENCE SIGNALS
    "lost deal because crm", "lost deal due to crm", "lost opportunity because crm",
    "crm mistake cost deal", "crm mistake cost opportunity",
    "deal slipped through cracks", "lead slipped through cracks",
    "pipeline forecast wrong", "forecast inaccurate crm", "forecast unreliable crm",
    "forecast wrong pipeline", "pipeline numbers wrong", "pipeline unreliable",
    "pipeline inaccurate crm", "sales forecast wrong", "sales forecast unreliable",
    "sales forecast inaccurate", "manager angry about crm", "manager complaining crm",
    "manager complaining pipeline", "manager complaining forecast",
    "manager asking pipeline updates", "manager asking crm updates",
    "manager chasing crm updates", "manager chasing pipeline updates",
    "crm data unreliable", "crm data inaccurate", "crm data outdated",
    "crm data incomplete", "crm data missing",

    # LEVEL 1 — CASUAL COMPLAINTS
    "updating crm is annoying", "crm admin work sucks", "crm updates are frustrating",
    "crm admin boring", "crm admin repetitive", "crm updates feel pointless",
    "crm updates slow me down", "crm admin work frustrating", "crm updates frustrating",
    "sales reps hate crm updates", "crm entry boring work", "crm entry tedious",
    "crm admin annoying", "updating salesforce annoying", "hubspot updates annoying",
    "zoho crm updates annoying",

    # LEVEL 2 — REPEATED WORKFLOW ANNOYANCE
    "update crm after every call", "update crm after meetings", "update crm end of day",
    "update crm daily", "enter crm notes after calls", "log calls after every call",
    "log meetings into crm", "log emails manually crm", "crm entry after each meeting",
    "crm updates take time", "crm updates slow workflow", "crm updates interrupt selling",
    "crm updates interrupt work", "crm admin taking time", "crm updates before leaving work",
    "crm updates weekly", "crm entries daily", "crm updates repetitive daily",
    "crm admin daily task", "crm updates constant", "crm updates every meeting",

    # LEVEL 3 — WORKFLOW FRICTION
    "spend time updating crm", "spend hours updating crm", "crm admin taking too much time",
    "crm data entry taking time", "crm work slowing sales", "crm updates slow productivity",
    "crm updates wasting time", "sales reps doing crm admin",
    "manually updating deals", "manual deal tracking", "manual lead tracking",
    "manual follow up tracking", "crm entry backlog", "crm admin workload",
    "crm admin heavy work", "crm updates exhausting", "crm updates draining time",
    "crm admin too much work", "crm admin killing productivity",

    # LEVEL 4 — OPERATIONAL DAMAGE
    "pipeline not updated", "deals not updated crm", "sales pipeline messy",
    "sales forecast wrong", "lost deal because crm", "lost opportunity because crm",
    "missed follow up lead", "forgot follow up lead",
    "pipeline data outdated", "pipeline messy crm",

    # LEVEL 5 — BUDGET / TOOL SEEKING
    "need crm automation", "tool to automate crm updates", "crm auto update tool",
    "automatic call logging crm", "automatic activity capture crm",
    "crm automation solution", "crm workflow automation tool",
    "sales automation crm logging", "crm automation software", "crm activity capture tool",
    "sales call auto logging", "auto update pipeline crm", "automate crm admin work",
    "automate crm updates", "crm automation for sales reps", "crm workflow automation",
    "crm automation for follow ups", "crm automation for pipeline",
    "crm automation for activities", "crm automation tool sales",

    # HIDDEN PROBLEM — FOLLOW-UP FAILURE
    "missed follow up opportunity", "follow ups slipping", "lost lead forgot follow up",
    "follow ups backlog", "follow ups hard to manage",

    # HIDDEN PROBLEM — PIPELINE VISIBILITY
    "pipeline visibility problem", "pipeline visibility poor", "pipeline hard to track",
    "pipeline messy sales team", "pipeline hard to maintain", "pipeline never accurate",

    # HIDDEN PROBLEM — FORECAST ACCURACY
    "sales forecast inaccurate", "forecast numbers wrong", "forecast unreliable pipeline",
    "forecast inaccurate deals", "forecast inaccurate crm data",
    "sales forecast messy", "forecast hard to trust", "forecast numbers off",
    "forecast unreliable sales",

    # HIDDEN PROBLEM — ACTIVITY TRACKING COMPLIANCE
    "sales reps forget to log calls", "sales team not updating crm",
    "team forgets crm updates", "sales reps ignore crm", "sales team ignores crm",
    "sales reps skip crm updates", "crm compliance problem sales",
    "sales team not logging activity", "activity logging inconsistent",
    "activity logging forgotten",

    # HIDDEN PROBLEM — SALES MANAGER VISIBILITY
    "manager asking crm updates", "manager chasing pipeline updates",
    "manager complaining pipeline", "manager complaining crm",
    "manager asking forecast numbers", "manager asking pipeline status",
    "manager chasing sales updates", "manager chasing crm entries",
    "manager wants pipeline updates", "manager asking deal updates",
]))

# ── HELPERS ───────────────────────────────────────────────────

def fmt_date(ts):
    try:
        return datetime.fromtimestamp(int(ts), tz=timezone.utc).strftime("%Y-%m-%d")
    except:
        return ""

def build_reddit_url(post):
    subreddit = post.get("subreddit", "")
    post_id   = post.get("id", "")
    slug      = post.get("title", "").lower().replace(" ", "_")[:50]
    if post_id:
        return f"https://www.reddit.com/r/{subreddit}/comments/{post_id}/{slug}/"
    return ""

def estimate_downvotes(score, upvote_ratio):
    try:
        ratio = float(upvote_ratio or 1)
        if ratio == 0 or ratio == 1:
            return 0
        upvotes   = round(score / (2 * ratio - 1))
        downvotes = max(0, upvotes - score)
        return downvotes
    except:
        return 0

def fetch_with_retry(url, headers, params, max_retries=5):
    """Retries on 429 with exponential backoff. Returns (resp, data) or (None, None)."""
    wait = 5
    for attempt in range(max_retries):
        resp = requests.get(url, headers=headers, params=params, timeout=15)
        if resp.status_code == 200:
            return resp, resp.json()
        elif resp.status_code == 429:
            print(f"    ⏳ 429 rate limit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
            time.sleep(wait)
            wait *= 2
        else:
            print(f"    ❌ HTTP {resp.status_code} | r/{params.get('subreddit')} | '{params.get('query','')[:50]}'")
            return None, None
    print(f"    💀 Gave up after {max_retries} retries on '{params.get('query','')[:50]}'")
    return None, None

# ── BULK SEARCH → CSV ─────────────────────────────────────────

CSV_FILE   = "crm_pain_research.csv"
CSV_FIELDS = ["title", "post_content", "upvotes", "downvotes",
              "url", "date", "subreddit", "author"]

DATE_AFTER  = "2025-09-01"
DATE_BEFORE = "2026-03-12"

seen_ids = set()
total_written = 0

SUBREDDITS = [
    "sales", "SaaS", "CRM", "salesforce", "hubspot",
    "startups", "entrepreneur", "smallbusiness", "B2B",
    "sales_advice", "marketing", "productivity",
]

with open(CSV_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
    writer.writeheader()

    for idx, query in enumerate(SEARCH_QUERIES, 1):
        for subreddit in SUBREDDITS:
            params = {
                "query"     : query,
                "subreddit" : subreddit,
                "after"     : DATE_AFTER,
                "before"    : DATE_BEFORE,
                "limit"     : 100,
                "sort"      : "desc",
            }

            resp, data = fetch_with_retry(f"{BASE}/posts/search", HEADERS, params)
            if data is None:
                continue

            posts = data.get("data") or []
            new_this_query = 0

            for post in posts:
                post_id = post.get("id", "")
                if not post_id or post_id in seen_ids:
                    continue
                seen_ids.add(post_id)

                score        = post.get("score", 0) or 0
                upvote_ratio = post.get("upvote_ratio", 1.0) or 1.0
                upvotes      = score
                downvotes    = estimate_downvotes(score, upvote_ratio)

                writer.writerow({
                    "title"        : post.get("title", ""),
                    "post_content" : post.get("selftext", ""),
                    "upvotes"      : upvotes,
                    "downvotes"    : downvotes,
                    "url"          : build_reddit_url(post),
                    "date"         : fmt_date(post.get("created_utc", 0)),
                    "subreddit"    : post.get("subreddit", ""),
                    "author"       : post.get("author", ""),
                })
                new_this_query += 1
                total_written  += 1

            if new_this_query:
                print(f"[{idx:>3}/{len(SEARCH_QUERIES)}] r/{subreddit:<15} | '{query[:45]:<45}' → {new_this_query:>3} new | total: {total_written}")

            time.sleep(1.5)

print(f"\n✅ Done. {total_written} unique posts saved to '{CSV_FILE}'.")


# ══════════════════════════════════════════════════════════════
# ORIGINAL DEMO SEARCHES (unchanged)
# ══════════════════════════════════════════════════════════════

submission_params = {
    "query"     : "saas founder",
    "subreddit" : "startups",
    "after"     : "2025-09-12",
    "before"    : "2026-03-12",
    "limit"     : 100,
    "sort"      : "desc",
}

resp = requests.get(
    f"{BASE}/posts/search",
    headers=HEADERS,
    params=submission_params
)
data = resp.json()
print(f"\nURL: {resp.url}\n")

for i, s in enumerate(data.get("data", [])):
    print(f"[{i+1}] {fmt_date(s.get('created_utc',0))} | Score: {s.get('score')} | {s.get('title','')[:80]}")
    print(f"     r/{s.get('subreddit')} | by {s.get('author')}")
    print()

comment_params = {
    "body"      : "churn mrr saas",
    "subreddit" : "SaaS",
    "after"     : "2025-09-12",
    "before"    : "2026-03-12",
    "limit"     : 100,
    "sort"      : "desc",
}

resp = requests.get(
    f"{BASE}/comments/search",
    headers=HEADERS,
    params=comment_params
)
data = resp.json()
print(f"URL: {resp.url}\n")

for i, c in enumerate(data.get("data", [])):
    body = c.get("body", "")
    print(f"[{i+1}] {fmt_date(c.get('created_utc',0))} | r/{c.get('subreddit')} | {c.get('author')}")
    print(f"     {body[:150]}{'...' if len(body) > 150 else ''}")
    print()

[  1/367] r/SaaS            | 'manual activity logging                      ' →   2 new | total: 2
[  1/367] r/CRM             | 'manual activity logging                      ' →   3 new | total: 5
[  1/367] r/salesforce      | 'manual activity logging                      ' →   1 new | total: 6
[  1/367] r/hubspot         | 'manual activity logging                      ' →   1 new | total: 7
[  1/367] r/smallbusiness   | 'manual activity logging                      ' →   1 new | total: 8
[  2/367] r/SaaS            | 'tool to automate crm updates                 ' →   8 new | total: 16
[  2/367] r/CRM             | 'tool to automate crm updates                 ' →   1 new | total: 17
[  2/367] r/hubspot         | 'tool to automate crm updates                 ' →   4 new | total: 21
[  2/367] r/entrepreneur    | 'tool to automate crm updates                 ' →   1 new | total: 22
[  2/367] r/smallbusiness   | 'tool to automate crm updates                 ' →   2 new | total: 24
[  4/